# 02 — ROI Detection: U-Net Segmentation

Binary segmentation of the ROI (reading window) using U-Net with ResNet34 encoder.
Predicted mask is converted to bounding box for IoU comparison with detection-based methods.
Two experiments: utility-meter dataset (bbox -> mask) and waterMeterDataset (polygon -> mask).

In [ ]:
import sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB:
    try:
        from google.colab import drive
        IN_COLAB = True
    except ImportError:
        pass

if IN_COLAB:
    from google.colab import drive, userdata

    drive.mount("/content/drive")

    token = userdata.get("GITHUB_TOKEN", "")
    base = f"https://{token}@github.com" if token else "https://github.com"
    if not Path("/content/WaterMeterCV").exists():
        subprocess.run(
            ["git", "clone", f"{base}/UrranQx/WaterMeterCV.git", "/content/WaterMeterCV"],
            check=True
        )

    BRANCH = "feature/roi-detection"
    subprocess.run(["git", "-C", "/content/WaterMeterCV", "checkout", BRANCH], check=True)

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "segmentation-models-pytorch", "albumentations",
         "rapidfuzz", "shapely"],
        check=True
    )

    ROOT         = Path("/content/WaterMeterCV")
    DATA_ROOT    = Path("/content/drive/MyDrive/WaterMeterCV/WaterMetricsDATA")
    WEIGHTS_BASE = Path("/content/drive/MyDrive/WaterMeterCV/weights")
    RESULTS_DIR  = Path("/content/drive/MyDrive/WaterMeterCV/results")
    WORKERS = 2
else:
    ROOT         = Path("../..").resolve()
    DATA_ROOT    = ROOT / "WaterMetricsDATA"
    WEIGHTS_BASE = ROOT / "models/weights"
    RESULTS_DIR  = ROOT / "results"
    WORKERS = 0

sys.path.insert(0, str(ROOT))
WEIGHTS_BASE.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import json
import time
import csv
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import cv2
import numpy as np
from datetime import datetime
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

from models.data.unified_loader import (
    load_utility_meter_dataset,
    load_water_meter_dataset_split,
)
from models.data.roi_dataset import polygon_to_bbox, filter_utility_meter_roi_samples
from models.metrics.evaluation import compute_iou_bbox

print(f"ROOT: {ROOT}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
UM_YOLO_PATH = DATA_ROOT / "utility-meter-reading-dataset-for-automatic-reading-yolo.v4i.yolov11"
WM_PATH = DATA_ROOT / "waterMeterDataset/WaterMeters"

WEIGHTS_DIR = WEIGHTS_BASE / "roi_unet"
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 256
BATCH_SIZE = 16
EPOCHS = 30
LR = 1e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

RUN_NAME = f"unet_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
print(f"Device: {DEVICE}, Run: {RUN_NAME}")

In [ ]:
class ROISegmentationDataset(Dataset):
    """Binary segmentation dataset: image + ROI mask."""

    def __init__(self, samples, img_size=256, transform=None, source="bbox"):
        """
        Args:
            samples: list of (image_path, roi_bbox) tuples OR UnifiedSample objects
            source: "bbox" — samples are (path, bbox) tuples
                    "polygon" — samples are UnifiedSample with roi_polygon
        """
        self.img_size = img_size
        self.transform = transform
        self.source = source
        if source == "bbox":
            self.items = [(p, b) for p, b in samples]
        else:
            self.items = [(s.image_path, s.roi_polygon) for s in samples
                          if s.roi_polygon is not None]

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        path, roi = self.items[idx]
        img = cv2.imread(str(path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]

        mask = np.zeros((h, w), dtype=np.uint8)
        if self.source == "bbox":
            cx, cy, bw, bh = roi
            x1 = int((cx - bw / 2) * w)
            y1 = int((cy - bh / 2) * h)
            x2 = int((cx + bw / 2) * w)
            y2 = int((cy + bh / 2) * h)
            mask[max(0, y1):y2, max(0, x1):x2] = 1
        else:  # polygon
            pts = np.array([(int(x * w), int(y * h)) for x, y in roi], dtype=np.int32)
            cv2.fillPoly(mask, [pts], 1)

        img = cv2.resize(img, (self.img_size, self.img_size))
        mask = cv2.resize(mask, (self.img_size, self.img_size),
                          interpolation=cv2.INTER_NEAREST)

        if self.transform:
            transformed = self.transform(image=img, mask=mask)
            img, mask = transformed["image"], transformed["mask"]

        return img, mask.unsqueeze(0).float()


train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, rotate_limit=15, p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

print("Dataset class and transforms ready.")

In [ ]:
def train_unet(train_dataset, val_dataset, tag, epochs=EPOCHS):
    """Train U-Net and return path to best weights."""
    model = smp.Unet(
        encoder_name="resnet34",
        encoder_weights="imagenet",
        in_channels=3,
        classes=1,
    ).to(DEVICE)

    bce = nn.BCEWithLogitsLoss()
    dice = smp.losses.DiceLoss(mode="binary", from_logits=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                              shuffle=True, num_workers=WORKERS)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE,
                            shuffle=False, num_workers=WORKERS)

    best_val_loss = float("inf")
    best_path = WEIGHTS_DIR / f"{tag}_best.pt"

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for imgs, masks in train_loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            preds = model(imgs)
            loss = bce(preds, masks) + dice(preds, masks)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for imgs, masks in val_loader:
                imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
                preds = model(imgs)
                val_loss += (bce(preds, masks) + dice(preds, masks)).item()

        avg_train = train_loss / len(train_loader)
        avg_val = val_loss / len(val_loader) if len(val_loader) > 0 else 0.0

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            torch.save(model.state_dict(), best_path)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1}/{epochs} — train: {avg_train:.4f}, val: {avg_val:.4f}")

    print(f"  Best val loss: {best_val_loss:.4f}, saved to {best_path}")
    return best_path


def mask_to_bbox_normalized(mask_np):
    """Convert binary mask (H, W) to (cx, cy, w, h) normalized bbox. Returns None if empty."""
    ys, xs = np.where(mask_np > 0)
    if len(xs) == 0:
        return None
    h, w = mask_np.shape
    x1, x2 = xs.min(), xs.max()
    y1, y2 = ys.min(), ys.max()
    return (
        (x1 + x2) / 2 / w,
        (y1 + y2) / 2 / h,
        (x2 - x1) / w,
        (y2 - y1) / h,
    )


def eval_unet(model, dataset, gt_bboxes):
    """Evaluate model. Returns (ious, inference_times_ms)."""
    model.eval()
    ious, times_ms = [], []

    for i in range(len(dataset)):
        img_tensor, _ = dataset[i]
        img_batch = img_tensor.unsqueeze(0).to(DEVICE)

        t0 = time.perf_counter()
        with torch.no_grad():
            pred = model(img_batch)
        times_ms.append((time.perf_counter() - t0) * 1000)

        pred_mask = (torch.sigmoid(pred[0, 0]).cpu().numpy() > 0.5).astype(np.uint8)
        pred_bbox = mask_to_bbox_normalized(pred_mask)

        if pred_bbox is not None and gt_bboxes[i] is not None:
            ious.append(compute_iou_bbox(pred_bbox, gt_bboxes[i]))
        else:
            ious.append(0.0)

    return ious, times_ms


print("Helper functions ready.")

In [ ]:
um_train_roi = filter_utility_meter_roi_samples(UM_YOLO_PATH, "train")
um_test_roi  = filter_utility_meter_roi_samples(UM_YOLO_PATH, "test")
print(f"UM ROI: {len(um_train_roi)} train, {len(um_test_roi)} test")

wm_train, wm_test = load_water_meter_dataset_split(WM_PATH, train_ratio=0.7, seed=42)
print(f"WM: {len(wm_train)} train, {len(wm_test)} test")

## Experiment 1: utility-meter dataset

In [ ]:
um_train_ds = ROISegmentationDataset(um_train_roi, IMG_SIZE, train_transform, source="bbox")
um_test_ds  = ROISegmentationDataset(um_test_roi,  IMG_SIZE, val_transform,   source="bbox")

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Training U-Net on UM...")
um_best_path = train_unet(um_train_ds, um_test_ds, f"um_{RUN_NAME}")

um_model = smp.Unet(encoder_name="resnet34", encoder_weights=None,
                     in_channels=3, classes=1).to(DEVICE)
um_model.load_state_dict(torch.load(um_best_path, map_location=DEVICE, weights_only=True))

um_gt_bboxes = [bbox for _, bbox in um_test_roi]
um_ious, um_inference_times = eval_unet(um_model, um_test_ds, um_gt_bboxes)

um_mean_iou = np.mean(um_ious) if um_ious else 0.0
um_det_rate = sum(1 for v in um_ious if v > 0) / len(um_ious) if um_ious else 0.0
um_avg_ms   = np.mean(um_inference_times) if um_inference_times else 0.0

print(f"\nUM — Mean IoU:        {um_mean_iou:.4f}")
print(f"UM — Detection rate:  {um_det_rate:.4f}")
print(f"UM — Avg inference:   {um_avg_ms:.1f} ms/image")

## Experiment 2: waterMeterDataset

In [ ]:
wm_train_ds = ROISegmentationDataset(wm_train, IMG_SIZE, train_transform, source="polygon")
wm_test_ds  = ROISegmentationDataset(wm_test,  IMG_SIZE, val_transform,   source="polygon")

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Training U-Net on WM...")
wm_best_path = train_unet(wm_train_ds, wm_test_ds, f"wm_{RUN_NAME}")

wm_model = smp.Unet(encoder_name="resnet34", encoder_weights=None,
                     in_channels=3, classes=1).to(DEVICE)
wm_model.load_state_dict(torch.load(wm_best_path, map_location=DEVICE, weights_only=True))

wm_gt_bboxes = [polygon_to_bbox(s.roi_polygon) if s.roi_polygon else None for s in wm_test]
wm_ious, wm_inference_times = eval_unet(wm_model, wm_test_ds, wm_gt_bboxes)

wm_mean_iou = np.mean(wm_ious) if wm_ious else 0.0
wm_det_rate = sum(1 for v in wm_ious if v > 0) / len(wm_ious) if wm_ious else 0.0
wm_avg_ms   = np.mean(wm_inference_times) if wm_inference_times else 0.0

print(f"\nWM — Mean IoU:        {wm_mean_iou:.4f}")
print(f"WM — Detection rate:  {wm_det_rate:.4f}")
print(f"WM — Avg inference:   {wm_avg_ms:.1f} ms/image")

## Predictions

In [ ]:
def show_mask_predictions(model, dataset, axes_row, title_prefix):
    """Draw mask overlay + predicted bbox on a row of axes."""
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])

    for i, ax in enumerate(axes_row):
        if i >= len(dataset):
            ax.axis("off")
            continue
        img_tensor, _ = dataset[i]

        img_np = img_tensor.numpy().transpose(1, 2, 0)
        img_np = (img_np * std + mean).clip(0, 1)

        with torch.no_grad():
            pred = model(img_tensor.unsqueeze(0).to(DEVICE))
        pred_mask = (torch.sigmoid(pred[0, 0]).cpu().numpy() > 0.5).astype(np.uint8)

        overlay = img_np.copy()
        overlay[pred_mask > 0, 0] = np.minimum(overlay[pred_mask > 0, 0] + 0.3, 1.0)

        ax.imshow(overlay)
        ax.set_title(f"{title_prefix} #{i}", fontsize=10)
        ax.axis("off")


fig, axes = plt.subplots(2, 4, figsize=(20, 10))
show_mask_predictions(um_model, um_test_ds, axes[0], "UM")
show_mask_predictions(wm_model, wm_test_ds, axes[1], "WM")

plt.suptitle("U-Net ROI Segmentation — Red overlay = predicted mask", fontsize=14)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "roi_unet_predictions.png", dpi=150)
plt.close()
print("Saved to results/roi_unet_predictions.png")

## Comparison

In [ ]:
print(f"{'='*60}")
print(f"{'Metric':<25} {'utility-meter':>15} {'waterMeter':>15}")
print(f"{'='*60}")
print(f"{'Mean IoU':<25} {um_mean_iou:>15.4f} {wm_mean_iou:>15.4f}")
print(f"{'Detection rate':<25} {um_det_rate:>15.4f} {wm_det_rate:>15.4f}")
print(f"{'Avg inference (ms)':<25} {um_avg_ms:>15.1f} {wm_avg_ms:>15.1f}")
print(f"{'N test':<25} {len(um_test_roi):>15d} {len(wm_test):>15d}")
print(f"{'='*60}")

## Save Results

In [ ]:
metrics = {
    "method": "unet_segmentation",
    "utility_meter": {
        "mean_iou": round(um_mean_iou, 4),
        "detection_rate": round(um_det_rate, 4),
        "avg_inference_ms": round(um_avg_ms, 1),
        "n_train": len(um_train_roi),
        "n_test": len(um_test_roi),
    },
    "water_meter": {
        "mean_iou": round(wm_mean_iou, 4),
        "detection_rate": round(wm_det_rate, 4),
        "avg_inference_ms": round(wm_avg_ms, 1),
        "n_train": len(wm_train),
        "n_test": len(wm_test),
    },
    "config": {
        "encoder": "resnet34",
        "img_size": IMG_SIZE,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "lr": LR,
    },
    "run_date": datetime.now().isoformat(),
}

with open(RESULTS_DIR / "roi_unet_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

csv_path = RESULTS_DIR / "roi_comparison.csv"
csv_exists = csv_path.exists()
with open(csv_path, "a", newline="") as f:
    writer = csv.writer(f)
    if not csv_exists:
        writer.writerow([
            "method", "um_mean_iou", "um_detection_rate", "um_inference_ms",
            "wm_mean_iou", "wm_detection_rate", "wm_inference_ms", "run_date",
        ])
    writer.writerow([
        "unet_segmentation",
        round(um_mean_iou, 4), round(um_det_rate, 4), round(um_avg_ms, 1),
        round(wm_mean_iou, 4), round(wm_det_rate, 4), round(wm_avg_ms, 1),
        datetime.now().strftime("%Y-%m-%d %H:%M"),
    ])

print(f"Metrics -> {RESULTS_DIR / 'roi_unet_metrics.json'}")
print(f"CSV    -> {csv_path}")

## Conclusions

*(Fill after running)*

- **U-Net (utility-meter):** Mean IoU=..., Detection rate=...
- **U-Net (waterMeter):** Mean IoU=..., Detection rate=...
- **Inference:** ... ms/image
- **Next step:** compare with YOLO and Faster R-CNN